# Weather Processing (NOAA ISD-Lite)

Clean and standardize raw hourly weather data.

## Input
- `data/raw/weather/weather_2024_isd_lite_raw.parquet`

## Outputs
- `data/processed/weather/weather_2024_hourly.parquet`
- `data/reports/weather/weather_process_missingness_2024.csv`


## Step 0: Ensure project data (local or Hugging Face)

If `data/` is missing, the next cell pulls [ahiruuu/CIS-5450](https://huggingface.co/datasets/ahiruuu/CIS-5450). Use `FLIGHT_SKIP_HF=1` to disable download.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _p in [_here] + list(_here.parents):
    if (_p / "notebooks" / "project_data.py").exists():
        sys.path.insert(0, str(_p / "notebooks"))
        break

from project_data import ensure_project_data

ensure_project_data()

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from project_data import resolve_project_root

PROJECT_ROOT = resolve_project_root()
DATA_ROOT = Path(os.getenv("FLIGHT_DATA_DIR", PROJECT_ROOT / "data")).expanduser().resolve()
RAW_DIR = DATA_ROOT / "raw" / "weather"
PROCESSED_DIR = DATA_ROOT / "processed" / "weather"
REPORT_DIR = DATA_ROOT / "reports" / "weather"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

raw_path = RAW_DIR / "weather_2024_isd_lite_raw.parquet"
if not raw_path.exists():
    raise FileNotFoundError(f"Missing input: {raw_path}")

weather_raw = pd.read_parquet(raw_path)
print(f"Project root: {PROJECT_ROOT}")
print(f"Loaded raw weather rows: {len(weather_raw):,}")


Project root: /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450
Loaded raw weather rows: 438,299


In [2]:
w = weather_raw.copy()

# ISD-Lite missing sentinel
w = w.replace(-9999, np.nan)

# Unit scaling (stored in tenths)
w["air_temp"] = w["air_temp"] / 10
w["dew_point"] = w["dew_point"] / 10
w["sea_level_pres"] = w["sea_level_pres"] / 10
w["wind_speed"] = w["wind_speed"] / 10

# Trace precipitation: -1 -> 0.001 before scaling
w["precip_1h"] = w["precip_1h"].replace(-1, 0.001) / 10
w["precip_6h"] = w["precip_6h"].replace(-1, 0.001) / 10

w["obs_time"] = pd.to_datetime(
    w[["year", "month", "day", "hour"]].astype(str).agg("-".join, axis=1),
    format="%Y-%m-%d-%H",
    errors="coerce"
)

weather_cols = [
    "air_temp", "dew_point", "sea_level_pres", "wind_dir",
    "wind_speed", "sky_cover", "precip_1h", "precip_6h"
]

w = w.sort_values(["IATA", "obs_time"])
w[weather_cols] = w.groupby("IATA")[weather_cols].transform(lambda x: x.ffill(limit=3))

keep_cols = ["IATA", "obs_time"] + weather_cols
weather_hourly = w[keep_cols].copy()

out_path = PROCESSED_DIR / "weather_2024_hourly.parquet"
weather_hourly.to_parquet(out_path, index=False)

missingness = (weather_hourly.isna().mean() * 100).round(4)
missingness_df = pd.DataFrame({
    "column": missingness.index,
    "missing_pct": missingness.values,
})
missingness_path = REPORT_DIR / "weather_process_missingness_2024.csv"
missingness_df.to_csv(missingness_path, index=False)

print(f"Processed weather rows: {len(weather_hourly):,}")
print(f"Saved processed data: {out_path}")
print(f"Saved missingness report: {missingness_path}")
print("Missingness (%):")
print(missingness)


Processed weather rows: 438,299
Saved processed data: /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data/processed/weather/weather_2024_hourly.parquet
Saved missingness report: /Users/hazel/Desktop/26_spring/cis_5450/CIS-5450/data/reports/weather/weather_process_missingness_2024.csv
Missingness (%):
IATA               0.0000
obs_time           0.0000
air_temp           0.0000
dew_point          0.0000
sea_level_pres     0.3174
wind_dir           0.0532
wind_speed         0.0000
sky_cover          7.5264
precip_1h          4.2133
precip_6h         86.7328
dtype: float64
